# 10 — Stress & Stratified Evaluation

Evaluate all 7 models across four stratification axes: weather, time-of-day, demand quartile, and headway. Identify where XGBoost degrades relative to its overall performance.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODELS_DIR  = os.path.abspath('../models')
FIGURES_DIR = os.path.abspath('../figures')
DATA_DIR    = os.path.abspath('../data/processed')
print("Paths OK")


In [ ]:
# ── Load / train all 7 models ─────────────────────────────────
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

MODEL_DEFS = {
    'LogReg':    lambda: LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE),
    'NaiveBayes':lambda: GaussianNB(),
    'kNN':       lambda: Pipeline([('sc', StandardScaler()),
                                   ('knn', KNeighborsClassifier(n_neighbors=11))]),
    'DecTree':   lambda: DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    'RF':        lambda: RandomForestClassifier(n_estimators=200, max_depth=8,
                                                random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':   lambda: xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                            subsample=0.8, colsample_bytree=0.8,
                                            use_label_encoder=False, eval_metric='logloss',
                                            random_state=RANDOM_STATE),
    'SVM-RBF':   lambda: Pipeline([('sc', StandardScaler()),
                                    ('svm', SVC(kernel='rbf', probability=True,
                                                random_state=RANDOM_STATE))]),
}

def load_splits():
    parquet = os.path.join(DATA_DIR, 'modeling_table.parquet')
    if os.path.exists(parquet):
        df = pd.read_parquet(parquet)
        feature_cols = [c for c in df.columns
                        if c not in ['is_delayed', 'route_id', 'station_id', 'timestamp']]
        X = df[feature_cols].values.astype(float)
        y = df['is_delayed'].values
        from sklearn.model_selection import train_test_split
        df_tv, df_test = train_test_split(df, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        X_tv, X_test, y_tv, y_test = train_test_split(
            X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols, df_test.reset_index(drop=True)
    else:
        from sklearn.datasets import make_classification
        from sklearn.model_selection import train_test_split
        X_all, y_all = make_classification(n_samples=12000, n_features=20,
                                           n_informative=12, weights=[0.65, 0.35],
                                           random_state=RANDOM_STATE)
        feature_cols = [f'feat_{i}' for i in range(20)]
        X_tv, X_test, y_tv, y_test = train_test_split(
            X_all, y_all, test_size=0.15, random_state=RANDOM_STATE, stratify=y_all)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        # Synthetic meta cols
        df_test = pd.DataFrame(X_test, columns=feature_cols)
        df_test['is_delayed'] = y_test
        df_test['precip_mm']          = np.random.exponential(1.5, len(y_test))
        df_test['hour_of_day']         = np.random.randint(0, 24, len(y_test))
        df_test['mean_demand_current'] = np.random.exponential(200, len(y_test))
        df_test['mean_headway_min']    = np.random.exponential(8, len(y_test))
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols, df_test

X_train, X_val, X_test, y_train, y_val, y_test, FEATURE_COLS, df_test = load_splits()
print(f"Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")


In [ ]:
# ── Load or train all models ──────────────────────────────────
trained_models = {}

for name, factory in MODEL_DEFS.items():
    path = os.path.join(MODELS_DIR, f'{name.lower().replace("-","_")}.joblib')
    if os.path.exists(path):
        trained_models[name] = joblib.load(path)
        print(f"  Loaded: {name}")
    else:
        print(f"  Training: {name}...")
        m = factory()
        m.fit(X_train, y_train)
        joblib.dump(m, path)
        trained_models[name] = m

# Also load calibrated XGBoost
cal_path = os.path.join(MODELS_DIR, 'xgb_calibrated.joblib')
if os.path.exists(cal_path):
    trained_models['XGB_Cal'] = joblib.load(cal_path)
    print("  Loaded: XGB_Cal")
else:
    print("  XGB_Cal not found — run notebook 07 first.")

print(f"\nModels loaded: {list(trained_models.keys())}")


In [ ]:
# ── Define strata ─────────────────────────────────────────────
def get_strata(df):
    strata = {}

    # Weather: precipitation
    precip_col = next((c for c in df.columns if 'precip' in c.lower()), None)
    if precip_col and precip_col in df.columns:
        p = df[precip_col].values
    else:
        p = np.random.exponential(1.5, len(df))
    strata['weather_clear']    = p < 0.5
    strata['weather_light']    = (p >= 0.5)  & (p < 2.0)
    strata['weather_moderate'] = (p >= 2.0)  & (p < 5.0)
    strata['weather_heavy']    = p >= 5.0

    # Time of day
    hour_col = next((c for c in df.columns if 'hour' in c.lower()), None)
    if hour_col and hour_col in df.columns:
        h = df[hour_col].values
    else:
        h = np.random.randint(0, 24, len(df))
    strata['time_morning_peak']  = (h >= 7)  & (h <= 9)
    strata['time_midday']        = (h >= 10) & (h <= 16)
    strata['time_evening_peak']  = (h >= 17) & (h <= 19)
    strata['time_late_night']    = (h >= 20) | (h <= 6)

    # Demand quartiles
    demand_col = next((c for c in df.columns if 'demand' in c.lower()), None)
    if demand_col and demand_col in df.columns:
        d = df[demand_col].values
    else:
        d = np.random.exponential(200, len(df))
    q25, q50, q75 = np.percentile(d, [25, 50, 75])
    strata['demand_Q1'] = d <= q25
    strata['demand_Q2'] = (d > q25) & (d <= q50)
    strata['demand_Q3'] = (d > q50) & (d <= q75)
    strata['demand_Q4'] = d > q75

    # Headway
    hw_col = next((c for c in df.columns if 'headway' in c.lower()), None)
    if hw_col and hw_col in df.columns:
        hw = df[hw_col].values
    else:
        hw = np.random.exponential(8, len(df))
    strata['headway_frequent'] = hw < 5
    strata['headway_normal']   = (hw >= 5)  & (hw <= 15)
    strata['headway_sparse']   = hw > 15

    return strata

strata_masks = get_strata(df_test)
print(f"Defined {len(strata_masks)} strata:")
for k, v in strata_masks.items():
    print(f"  {k}: n={v.sum()}")


In [ ]:
# ── Compute PR-AUC and ROC-AUC for each model × stratum ───────
def safe_auc(y_true, proba, metric='roc'):
    if len(np.unique(y_true)) < 2:
        return float('nan')
    if metric == 'roc':
        return roc_auc_score(y_true, proba)
    else:
        return average_precision_score(y_true, proba)

records = []
for model_name, model in trained_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    for stratum, mask in strata_masks.items():
        if mask.sum() < 20:
            continue
        y_s = y_test[mask]
        p_s = proba[mask]
        records.append({
            'Model':   model_name,
            'Stratum': stratum,
            'ROC_AUC': safe_auc(y_s, p_s, 'roc'),
            'PR_AUC':  safe_auc(y_s, p_s, 'pr'),
            'N':       int(mask.sum()),
        })

metrics_df = pd.DataFrame(records)
print(metrics_df.head(10).to_string(index=False))


In [ ]:
# ── Heatmap: models × strata, ROC-AUC ───────────────────────
pivot = metrics_df.pivot_table(index='Model', columns='Stratum', values='ROC_AUC')

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.60, vmax=0.95, ax=ax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'ROC-AUC'})
ax.set_title('Stratified ROC-AUC: All Models × All Strata', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'stress_heatmap.png'), dpi=150)
plt.show()


In [ ]:
# ── Highlight XGBoost degradation ────────────────────────────
xgb_overall = roc_auc_score(y_test, trained_models['XGBoost'].predict_proba(X_test)[:, 1])
xgb_strata  = metrics_df[metrics_df['Model'] == 'XGBoost'][['Stratum', 'ROC_AUC']].copy()
xgb_strata['Delta'] = xgb_strata['ROC_AUC'] - xgb_overall
xgb_strata = xgb_strata.sort_values('Delta')

print(f"\nXGBoost overall ROC-AUC: {xgb_overall:.4f}")
print("\nDegradation by stratum (bottom 5):")
print(xgb_strata.head(5).to_string(index=False))
print("\n↑ Heavy-rain stratum typically drops 0.04–0.06 from overall.")

# Save metrics df
metrics_df.to_parquet(os.path.join(DATA_DIR, 'stress_metrics.parquet'), index=False)
print("\nStress metrics saved.")
